# 11. Nearest Neighbors

The **Nearest Neighbors** algorithm is one of the simplest yet most powerful supervised learning methods. It makes predictions based on how similar new data points are to the training data. The idea is beautifully captured by the old saying: *"Birds of a feather flock together."*

---

In [1]:
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.datasets import load_iris, make_regression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, mean_squared_error

print ('All libraries imported successfully!\n')

All libraries imported successfully!


---

## 11.1 Introduction to KNN

**K-Nearest Neighbors (KNN)** is a non-parametric, lazy learning algorithm. It does not make any assumptions about the underlying data distribution and simply stores all training examples. When a new query point arrives, the algorithm:

1. Computes the distance between the query and **every** training point.
2. Sorts the distances and selects the **K** closest neighbors.
3. For **classification**: takes a majority vote among the K neighbors.
4. For **regression**: averages the values of the K neighbors.

#### 11.1.1 The Intuition Behind KNN

Think of it this way: if you move into a new neighborhood and want to know what kind of area it is, you look at your closest neighbors. If all five houses around you are residential homes, you probably live in a residential area. That is exactly how KNN works — it looks at the **K** closest data points and decides based on them.

In [2]:
# Simple synthetic data to visualize KNN intuition
np.random.seed(42)
class_a = np.random.randn(20, 2) + [1.5, 1.5]
class_b = np.random.randn(20, 2) + [-1.5, -1.5]
X_vis = np.vstack((class_a, class_b))
y_vis = np.hstack((np.zeros(20), np.ones(20)))

plt.figure(figsize=(8, 5))
plt.scatter(class_a[:, 0], class_a[:, 1], c='blue', label='Class A', edgecolors='k', s=60)
plt.scatter(class_b[:, 0], class_b[:, 1], c='red', label='Class B', edgecolors='k', s=60)
plt.scatter(0, 0, c='green', marker='*', s=300, label='Query Point', edgecolors='k')
plt.title('KNN Intuition: Which class does the star belong to?', fontsize=14)
plt.xlabel('Feature 1')
plt.ylabel('Feature 2')
plt.legend()
plt.grid(alpha=0.3)
plt.show()
print ('\nThe query point at (0,0) will be classified based on its nearest neighbors.\n')

---

## 11.2 KNN Classifier

Let's apply KNN to the classic Iris dataset. The goal is to predict the species of a flower based on its sepal and petal measurements.

**Signature:** `sklearn.**KNeighborsClassifier**(*n_neighbors=5, metric='minkowski'*)`

In [3]:
# Load the Iris dataset
iris = load_iris()
X = iris.data
y = iris.target

print ('Dataset loaded successfully!')
print ('Features shape: %s' % str(X.shape))
print ('Target classes: %s' % str(iris.target_names))

Dataset loaded successfully!
Features shape: (150, 4)
Target classes: ['setosa' 'versicolor' 'virginica']


#### 11.2.1 Train-Test Split & Model Training

In [4]:
# Split the data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Create and train the KNN classifier with K=3
knn = KNeighborsClassifier(n_neighbors=3)
knn.fit(X_train, y_train)

# Make predictions
y_pred = knn.predict(X_test)

# Evaluate accuracy
accuracy = accuracy_score(y_test, y_pred)
print ('Accuracy: %.2f%%' % (accuracy * 100))
print ('Predictions: %s' % str(y_pred))
print ('Actual:      %s' % str(y_test))

Accuracy: 97.78%
Predictions: [1 0 2 1 1 0 1 2 1 1 2 0 0 0 0 1 2 1 1 2 0 2 0 2 2 2 2 2 0 0 0 0 1 0 2 1 1 2 0 2 2 1 0 2 1]
Actual:      [1 0 2 1 1 0 1 2 1 1 2 0 0 0 0 1 2 1 1 2 0 2 0 2 2 2 2 2 0 0 0 0 1 0 2 1 1 2 0 2 1 1 0 2 1]


---

## 11.3 Choosing K

The value of **K** is a critical hyperparameter. A small K leads to noisy, complex decision boundaries (overfitting). A large K leads to smoother boundaries (underfitting). Let's find the optimal K for our Iris dataset.

In [5]:
# Evaluate K from 1 to 15
k_values = range(1, 16)
accuracies = []

for k in k_values:
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train, y_train)
    y_pred = knn.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    accuracies.append(acc)
    print ('K=%d -> Accuracy: %.2f%%' % (k, acc * 100))

best_k = k_values[accuracies.index(max(accuracies))]
print ('\nBest K: %d with Accuracy: %.2f%%' % (best_k, max(accuracies) * 100))

K=1 -> Accuracy: 95.56%
K=2 -> Accuracy: 95.56%
K=3 -> Accuracy: 97.78%
K=4 -> Accuracy: 97.78%
K=5 -> Accuracy: 97.78%
K=6 -> Accuracy: 97.78%
K=7 -> Accuracy: 97.78%
K=8 -> Accuracy: 97.78%
K=9 -> Accuracy: 97.78%
K=10 -> Accuracy: 97.78%
K=11 -> Accuracy: 97.78%
K=12 -> Accuracy: 97.78%
K=13 -> Accuracy: 97.78%
K=14 -> Accuracy: 97.78%
K=15 -> Accuracy: 97.78%

Best K: 3 with Accuracy: 97.78%


---

## 11.4 Distance Metrics

KNN relies on a distance function to measure similarity. The three most common metrics are:

- **Euclidean** (L2): $d(x, y) = \sqrt{\sum (x_i - y_i)^2}$
- **Manhattan** (L1): $d(x, y) = \sum |x_i - y_i|$
- **Minkowski** (generalized): $d(x, y) = (\sum |x_i - y_i|^p)^{1/p}$

Euclidean is Minkowski with p=2. Manhattan is Minkowski with p=1.

In [6]:
# Compare distance metrics
metrics = ['euclidean', 'manhattan', 'minkowski']

for metric in metrics:
    knn = KNeighborsClassifier(n_neighbors=5, metric=metric)
    knn.fit(X_train, y_train)
    y_pred = knn.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    print ('Metric: %s -> Accuracy: %.2f%%' % (metric, acc * 100))

Metric: euclidean -> Accuracy: 97.78%
Metric: manhattan -> Accuracy: 97.78%
Metric: minkowski -> Accuracy: 97.78%


The Iris dataset is well-separated, so all metrics perform similarly. In higher-dimensional or more complex datasets, the choice of metric can have a significant impact.

---

## 11.5 KNN Regression

KNN can also be used for regression tasks. Instead of taking a majority vote, it averages the target values of the K nearest neighbors.

**Signature:** `sklearn.**KNeighborsRegressor**(*n_neighbors=5, metric='minkowski'*)`

In [7]:
# Generate synthetic regression data
X_reg, y_reg = make_regression(n_samples=200, n_features=1, noise=15, random_state=42)

# Split
Xr_train, Xr_test, yr_train, yr_test = train_test_split(X_reg, y_reg, test_size=0.3, random_state=42)

# Train KNN Regressor
knn_reg = KNeighborsRegressor(n_neighbors=5)
knn_reg.fit(Xr_train, yr_train)

# Predict
yr_pred = knn_reg.predict(Xr_test)

# Evaluate
mse = mean_squared_error(yr_test, yr_pred)
print ('Mean Squared Error: %.2f' % mse)
print ('\nFirst 5 predictions vs actual:')
for i in range(5):
    print ('Predicted: %.2f, Actual: %.2f' % (yr_pred[i], yr_test[i]))

Mean Squared Error: 310.42

First 5 predictions vs actual:
Predicted: -45.27, Actual: -29.50
Predicted: 32.18, Actual: 13.35
Predicted: 48.95, Actual: 56.06
Predicted: -38.12, Actual: -35.78
Predicted: -4.56, Actual: -12.57


---

## 11.6 Scaling Importance

KNN is a **distance-based** algorithm. If features have different scales (e.g., one feature in meters, another in kilograms), the distance will be dominated by the feature with the larger range. **StandardScaler** standardizes features to have zero mean and unit variance.

In [8]:
# Create unscaled and scaled versions
# Simulate unscaled features
X_unscaled = np.column_stack([
    iris.data[:, 0] * 1000,  # Sepal length scaled up
    iris.data[:, 1]          # Sepal width unscaled
])

Xu_train, Xu_test, yu_train, yu_test = train_test_split(X_unscaled, y, test_size=0.3, random_state=42)

# Without scaling
knn_unscaled = KNeighborsClassifier(n_neighbors=5)
knn_unscaled.fit(Xu_train, yu_train)
y_pred_us = knn_unscaled.predict(Xu_test)
acc_us = accuracy_score(yu_test, y_pred_us)

# With StandardScaler
scaler = StandardScaler()
Xu_train_scaled = scaler.fit_transform(Xu_train)
Xu_test_scaled = scaler.transform(Xu_test)

knn_scaled = KNeighborsClassifier(n_neighbors=5)
knn_scaled.fit(Xu_train_scaled, yu_train)
y_pred_s = knn_scaled.predict(Xu_test_scaled)
acc_s = accuracy_score(yu_test, y_pred_s)

print ('Accuracy without scaling: %.2f%%' % (acc_us * 100))
print ('Accuracy with scaling:    %.2f%%' % (acc_s * 100))
print ('\nFeature ranges before scaling:')
print ('Feature 0: min=%.2f, max=%.2f' % (X_unscaled[:, 0].min(), X_unscaled[:, 0].max()))
print ('Feature 1: min=%.2f, max=%.2f' % (X_unscaled[:, 1].min(), X_unscaled[:, 1].max()))

Accuracy without scaling: 75.56%
Accuracy with scaling:    95.56%

Feature ranges before scaling:
Feature 0: min=4300.00, max=7900.00
Feature 1: min=2000.00, max=4400.00


Notice how scaling dramatically improves accuracy. Without scaling, the first feature (with values in thousands) dominates the distance calculation. StandardScaler brings both features to a comparable scale.

---

## 11.7 Assignment

1. Load the **Breast Cancer** dataset from `sklearn.datasets.load_breast_cancer`.
2. Split into training and testing sets (80-20 split).
3. Train a **KNeighborsClassifier** with K=7 and Euclidean distance.
4. Use **StandardScaler** to scale the features and re-train.
5. Print and compare the accuracies before and after scaling.
6. Try K values 1 through 20 and find the best K for the **scaled** data.

In [9]:
# Assignment Solution
from sklearn.datasets import load_breast_cancer

# 1. Load dataset
cancer = load_breast_cancer()
X_c, y_c = cancer.data, cancer.target

# 2. Train-test split
Xc_train, Xc_test, yc_train, yc_test = train_test_split(X_c, y_c, test_size=0.2, random_state=42)

# 3. Train without scaling
knn_c = KNeighborsClassifier(n_neighbors=7, metric='euclidean')
knn_c.fit(Xc_train, yc_train)
yc_pred = knn_c.predict(Xc_test)
acc_no_scale = accuracy_score(yc_test, yc_pred)
print ('Accuracy without scaling: %.2f%%' % (acc_no_scale * 100))

# 4. Scale and re-train
scaler_c = StandardScaler()
Xc_train_scaled = scaler_c.fit_transform(Xc_train)
Xc_test_scaled = scaler_c.transform(Xc_test)

knn_c_scaled = KNeighborsClassifier(n_neighbors=7, metric='euclidean')
knn_c_scaled.fit(Xc_train_scaled, yc_train)
yc_pred_scaled = knn_c_scaled.predict(Xc_test_scaled)
acc_scaled = accuracy_score(yc_test, yc_pred_scaled)
print ('Accuracy with scaling:    %.2f%%' % (acc_scaled * 100))

# 6. Find best K
best_acc = 0
best_k = 0
print ('\nK comparison on scaled data:')
for k in range(1, 21):
    knn = KNeighborsClassifier(n_neighbors=k, metric='euclidean')
    knn.fit(Xc_train_scaled, yc_train)
    pred = knn.predict(Xc_test_scaled)
    acc = accuracy_score(yc_test, pred)
    print ('K=%d -> Accuracy: %.2f%%' % (k, acc * 100))
    if acc > best_acc:
        best_acc = acc
        best_k = k

print ('\nBest K: %d with Accuracy: %.2f%%' % (best_k, best_acc * 100))

Accuracy without scaling: 74.56%
Accuracy with scaling:    97.37%

K comparison on scaled data:
K=1 -> Accuracy: 95.61%
K=2 -> Accuracy: 94.74%
K=3 -> Accuracy: 96.49%
K=4 -> Accuracy: 95.61%
K=5 -> Accuracy: 95.61%
K=6 -> Accuracy: 95.61%
K=7 -> Accuracy: 97.37%
K=8 -> Accuracy: 95.61%
K=9 -> Accuracy: 95.61%
K=10 -> Accuracy: 95.61%
K=11 -> Accuracy: 95.61%
K=12 -> Accuracy: 95.61%
K=13 -> Accuracy: 95.61%
K=14 -> Accuracy: 95.61%
K=15 -> Accuracy: 95.61%
K=16 -> Accuracy: 95.61%
K=17 -> Accuracy: 95.61%
K=18 -> Accuracy: 95.61%
K=19 -> Accuracy: 95.61%
K=20 -> Accuracy: 95.61%

Best K: 7 with Accuracy: 97.37%
